# The ambulance dispatcher's question

A crash has been reported somewhere in New Zealand. You are the ambulance
dispatcher, and you can ask only **one** question about it. Which one tells you
most about whether someone is badly hurt?

- **A.** Was it dark?
- **B.** Was it raining heavily?
- **C.** Was it on a 100 km/h road?
- **D.** Was a motorbike involved?

Vote now, before any code runs. Then we answer it with real NZ crash records,
first by hand and then by letting a decision tree do the search for us.


**About the data.** Every crash reported to Waka Kotahi by NZ Police goes into
the Crash Analysis System (CAS). This file holds the crashes from **2023**,
downloaded on 19 October 2026. CAS is updated every month and older records can
still change, so a fresh download may differ slightly.

Source: Waka Kotahi NZ Transport Agency, Crash Analysis System (CAS) data,
licensed CC BY 4.0.
https://www.nzta.govt.nz/safety/partners/crash-analysis-system/

## What does one crash record look like?

Each row is one crash. The columns we use:

| Column | What it holds |
|---|---|
| `crashSeverity` | the worst injury in the crash: Fatal, Serious, Minor or Non-Injury |
| `speedLimit` | the speed limit where it happened, in km/h |
| `light` | Bright sun, Overcast, Twilight, Dark, or Unknown |
| `weatherA` | Fine, Light rain, Heavy rain, Mist or Fog, Snow, Hail or Sleet |
| `roadSurface` | Sealed, Unsealed, End of seal |
| `urban` | Urban or Open (open road). In this file it follows the speed limit: 70 km/h and below is always Urban |
| `flatHill` | Flat or Hill Road |
| `NumberOfLanes` | lanes on the road |
| `motorcycle`, `bicycle`, `truck` | how many were involved (0 means none) |

`Null` and `Unknown` mean the police report did not record it.

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

crashes = pd.read_csv("../../../data/nz_crashes_2023.csv")

print("Rows and columns:", crashes.shape)
crashes.head()

Rows and columns: (31511, 12)


,crashYear,crashSeverity,speedLimit,light,weatherA,roadSurface,urban,flatHill,NumberOfLanes,motorcycle,bicycle,truck
0,2023,Minor Crash,50.0,Overcast,Light rain,Sealed,Urban,Flat,2.0,0,0,0
1,2023,Minor Crash,100.0,Twilight,Fine,Sealed,Open,Flat,2.0,0,0,0
2,2023,Minor Crash,50.0,Overcast,Fine,Sealed,Urban,Flat,1.0,0,0,0
3,2023,Minor Crash,100.0,Bright sun,Fine,Sealed,Open,Flat,2.0,0,0,1
4,2023,Minor Crash,80.0,Dark,Fine,Sealed,Open,Flat,2.0,0,0,0


In [ ]:
print("Crashes before:", len(crashes))

crashes = crashes.dropna(subset=["speedLimit", "NumberOfLanes"])

print("Crashes after: ", len(crashes))

In [2]:
# Count how many crashes there are of each severity
# Hint: the column is "crashSeverity", and .value_counts() does the counting
crashes["crashSeverity"].value_counts()

crashSeverity
Non-Injury Crash    20549
Minor Crash          8571
Serious Crash        2086
Fatal Crash           305
Name: count, dtype: int64

In [3]:
# Hint: copy the two labels exactly as they appear in the counts above
is_serious = crashes["crashSeverity"].isin(["Serious Crash", "Fatal Crash"])
crashes["serious"] = is_serious.astype(int)

print("Serious crashes:", crashes["serious"].sum())
print("Share serious:  ", round(crashes["serious"].mean(), 3))

Serious crashes: 2391
Share serious:   0.076


In [4]:
TEXT_COLUMNS = ["light", "weatherA", "roadSurface", "urban", "flatHill"]
NUMBER_COLUMNS = ["speedLimit", "NumberOfLanes", "motorcycle", "bicycle", "truck"]

# dtype=int gives 0 and 1 instead of False and True
X = pd.get_dummies(crashes[NUMBER_COLUMNS + TEXT_COLUMNS], columns=TEXT_COLUMNS, dtype=int)
y = crashes["serious"]

print("Rows and columns in X:", X.shape)
print(list(X.columns))

Rows and columns in X: (31511, 26)
['speedLimit', 'NumberOfLanes', 'motorcycle', 'bicycle', 'truck', 'light_Bright sun', 'light_Dark', 'light_Overcast', 'light_Twilight', 'light_Unknown', 'weatherA_Fine', 'weatherA_Hail or Sleet', 'weatherA_Heavy rain', 'weatherA_Light rain', 'weatherA_Mist or Fog', 'weatherA_Null', 'weatherA_Snow', 'roadSurface_End of seal', 'roadSurface_Null', 'roadSurface_Sealed', 'roadSurface_Unsealed', 'urban_Open', 'urban_Urban', 'flatHill_Flat', 'flatHill_Hill Road', 'flatHill_Null']


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,     # Hint: a quarter, as a decimal
    stratify=y,      # Hint: the variable that holds the 0/1 target
    random_state=42)

print("Training crashes:", len(X_train), "  share serious:", round(y_train.mean(), 3))
print("Test crashes:    ", len(X_test), "  share serious:", round(y_test.mean(), 3))

Training crashes: 23633   share serious: 0.076
Test crashes:     7878   share serious: 0.076


In [6]:
one_question_tree = DecisionTreeClassifier(max_depth=1, random_state=42)   # Hint: one question
one_question_tree.fit(X_train, y_train)                                              # Hint: the training data

print(export_text(one_question_tree, feature_names=list(X.columns), show_weights=True))

|--- motorcycle <= 0.50
|   |--- weights: [21142.00, 1372.00] class: 0
|--- motorcycle >  0.50
|   |--- weights: [698.00, 421.00] class: 0



In [7]:
predictions = one_question_tree.predict(X_test)    # Hint: the test crashes

print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print("Recall:  ", round(recall_score(y_test, predictions), 3))
print(confusion_matrix(y_test, predictions))

Accuracy: 0.924
Recall:   0.0
[[7280    0]
 [ 598    0]]


How to read the confusion matrix: rows are what really happened, columns are
what the tree predicted, in the order not serious (0), then serious (1).

|  | predicted not serious | predicted serious |
|---|---|---|
| **really not serious** | correct | false alarm |
| **really serious** | **missed** | caught |

**Recall** is the share of really serious crashes that the tree caught:

$$\text{recall} = \frac{\text{caught}}{\text{caught} + \text{missed}}$$

Would you buy this tree for the ambulance service?

In [11]:
balanced_tree = DecisionTreeClassifier(max_depth=1, class_weight="balanced", random_state=42)   # Hint: a word, in quotes
balanced_tree.fit(X_train, y_train)

print(export_text(balanced_tree, feature_names=list(X.columns), show_weights=True))

predictions = balanced_tree.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print("Recall:  ", round(recall_score(y_test, predictions), 3))
print(confusion_matrix(y_test, predictions))

|--- motorcycle <= 0.50
|   |--- weights: [11438.85, 9041.96] class: 0
|--- motorcycle >  0.50
|   |--- weights: [377.65, 2774.54] class: 1

Accuracy: 0.908
Recall:   0.206
[[7031  249]
 [ 475  123]]
